# 1.3 · Versioning, feature lifecycle and what changed in 2026-07-28

**版本、特性生命周期与 2026-07-28 的变化**

> **After this lesson you can:** Negotiate protocol versions per request, explain how dual-era clients and servers interoperate with 2025-era MCP, and list every major change and deprecation between 2025-11-25 and 2026-07-28.

Official reading: [Versioning](https://modelcontextprotocol.io/docs/2026-07-28/learn/versioning) · [Spec: Versioning and compatibility](https://modelcontextprotocol.io/specification/2026-07-28/basic/versioning) · [Key changes (changelog)](https://modelcontextprotocol.io/specification/2026-07-28/changelog) · [Deprecated features](https://modelcontextprotocol.io/specification/2026-07-28/deprecated) · [Feature lifecycle policy](https://modelcontextprotocol.io/community/feature-lifecycle)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. Version strings and a negotiation function
Versions are an enumerated set. A client and a server each support some subset; the client wants the newest version both share. Order comes from an explicit registry, not from string comparison.

In [ ]:
import json

KNOWN_VERSIONS = ["2024-11-05", "2025-03-26", "2025-06-18", "2025-11-25", "2026-07-28"]  # oldest -> newest
MODERN = {"2026-07-28"}   # per-request _meta era; everything older uses the initialize handshake


def choose_version(client_supports, server_supports, known=KNOWN_VERSIONS):
    """Newest version both sides support, or None. Unknown strings are ignored, never 'newer'."""
    mutual = [v for v in known if v in client_supports and v in server_supports]
    return mutual[-1] if mutual else None


cases = {
    "both modern":               (["2026-07-28", "2025-11-25"], ["2026-07-28"]),
    "server still on 2025":      (["2026-07-28", "2025-11-25"], ["2025-06-18", "2025-11-25"]),
    "no overlap":                (["2026-07-28"], ["2025-03-26"]),
    "server claims 'zzz'":       (["2026-07-28"], ["zzz", "2026-07-28"]),
}
for label, (client, server) in cases.items():
    picked = choose_version(client, server)
    era = "-" if picked is None else ("modern" if picked in MODERN else "legacy handshake")
    print(f"{label:<22} -> {picked}  ({era})")

### 2. Three servers for the rest of the notebook
We need one server of each era. The SDK's `MCPServer` is **dual-era**. The other two are tiny hand-written JSON-RPC loops: a **legacy** server that only understands `initialize` (like any 2025 server that hasn't upgraded), and a **modern-only** server that rejects `initialize` with `-32022` naming its versions, as the spec recommends. A small **wiretap** script sits between client and server and logs every line in both directions.

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

workdir = Path(tempfile.mkdtemp())

(workdir / "dual_server.py").write_text(textwrap.dedent('''
    from mcp.server import MCPServer
    server = MCPServer("weather", version="2.3.0")

    @server.tool()
    def weather_current(location: str) -> str:
        """Get current weather for a location."""
        return f"{location}: 20 degrees (dual-era SDK server)"

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))

(workdir / "legacy_server.py").write_text(textwrap.dedent('''
    """A 2025-11-25-era server: initialize handshake or nothing."""
    import json, sys
    initialized = False
    def reply(id_, result=None, error=None):
        print(json.dumps({"jsonrpc": "2.0", "id": id_, **({"error": error} if error else {"result": result})}), flush=True)
    for line in sys.stdin:
        msg = json.loads(line); method, id_ = msg.get("method"), msg.get("id")
        if method == "initialize":
            reply(id_, {"protocolVersion": "2025-11-25", "capabilities": {"tools": {}},
                        "serverInfo": {"name": "old-weather", "version": "0.9.0"}})
        elif method == "notifications/initialized":
            initialized = True
        elif not initialized:
            reply(id_, error={"code": -32600, "message": "Server not initialized"})
        elif method == "tools/list":
            reply(id_, {"tools": [{"name": "weather_current", "description": "Current weather",
                                   "inputSchema": {"type": "object", "properties": {"location": {"type": "string"}}}}]})
        elif method == "tools/call":
            reply(id_, {"content": [{"type": "text", "text": "Sunny (legacy 2025 server)"}]})
        elif method == "ping":
            reply(id_, {})
        elif id_ is not None:
            reply(id_, error={"code": -32601, "message": f"Method not found: {method}"})
'''))

(workdir / "modern_only_server.py").write_text(textwrap.dedent('''
    """A server that speaks 2026-07-28 and nothing else."""
    import json, sys
    SUPPORTED = ["2026-07-28"]
    INFO = {"io.modelcontextprotocol/serverInfo": {"name": "new-weather", "version": "3.0.0"}}
    def send(msg): print(json.dumps({"jsonrpc": "2.0", **msg}), flush=True)
    for line in sys.stdin:
        msg = json.loads(line); method, id_ = msg.get("method"), msg.get("id")
        if id_ is None:
            continue
        if method == "initialize":   # spec: name supported versions; legacy clients cannot fall forward
            send({"id": id_, "error": {"code": -32022, "message": "This server only speaks 2026-07-28",
                  "data": {"supported": SUPPORTED, "requested": msg["params"].get("protocolVersion")}}})
            continue
        version = msg.get("params", {}).get("_meta", {}).get("io.modelcontextprotocol/protocolVersion")
        if version is None:
            send({"id": id_, "error": {"code": -32602, "message": "missing _meta protocolVersion"}})
        elif version not in SUPPORTED:
            send({"id": id_, "error": {"code": -32022, "message": "Unsupported protocol version",
                  "data": {"supported": SUPPORTED, "requested": version}}})
        elif method == "server/discover":
            send({"id": id_, "result": {"resultType": "complete", "supportedVersions": SUPPORTED,
                  "capabilities": {"tools": {}}, "ttlMs": 60000, "cacheScope": "public", "_meta": INFO}})
        elif method == "tools/list":
            send({"id": id_, "result": {"resultType": "complete", "ttlMs": 60000, "cacheScope": "public", "_meta": INFO,
                  "tools": [{"name": "weather_current", "description": "Current weather",
                             "inputSchema": {"type": "object", "properties": {"location": {"type": "string"}}}}]}})
        elif method == "tools/call":
            send({"id": id_, "result": {"resultType": "complete", "_meta": INFO,
                  "content": [{"type": "text", "text": "Sunny (modern-only server)"}]}})
        else:
            send({"id": id_, "error": {"code": -32601, "message": f"Method not found: {method}"}})
'''))

(workdir / "wiretap.py").write_text(textwrap.dedent('''
    """Usage: python wiretap.py LOGFILE server.py -- relays stdio and logs each line."""
    import subprocess, sys, threading
    log_path, target = sys.argv[1], sys.argv[2:]
    child = subprocess.Popen([sys.executable, *target], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
    lock = threading.Lock()
    def record(prefix, line):
        with lock, open(log_path, "a") as f:
            f.write(prefix + " " + line.rstrip("\\n") + "\\n")
    def pump():
        for line in child.stdout:
            record("<<<", line); sys.stdout.write(line); sys.stdout.flush()
    t = threading.Thread(target=pump, daemon=True); t.start()
    for line in sys.stdin:
        record(">>>", line); child.stdin.write(line); child.stdin.flush()
    child.stdin.close(); child.wait(); t.join(timeout=2)
'''))

SERVERS = {"dual-era": workdir / "dual_server.py",
           "legacy": workdir / "legacy_server.py",
           "modern-only": workdir / "modern_only_server.py"}
print("written:", sorted(p.name for p in workdir.iterdir()))

### 3. Per-request negotiation by hand: `-32022` and retry
A future client prefers a (hypothetical) newer revision `2027-03-01` and also speaks `2026-07-28`. It sends its request without any discovery, reads `data.supported` from the error, negotiates, and retries — exactly the flow in the versioning spec.

In [ ]:
CLIENT_SUPPORTS = ["2027-03-01", "2026-07-28"]   # newest first; 2027-03-01 is made up for the demo


def envelope(version):
    return {"io.modelcontextprotocol/protocolVersion": version,
            "io.modelcontextprotocol/clientCapabilities": {},
            "io.modelcontextprotocol/clientInfo": {"name": "future-client", "version": "1.0"}}


proc = subprocess.Popen([sys.executable, str(SERVERS["dual-era"])],
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)


def rpc(message):
    print(">>>", json.dumps(message)[:200])
    proc.stdin.write(json.dumps(message) + "\n"); proc.stdin.flush()
    reply = json.loads(proc.stdout.readline())
    print("<<<", json.dumps(reply)[:300], "\n")
    return reply


def call_with_negotiation(request_id, method, params):
    version = CLIENT_SUPPORTS[0]
    for attempt in range(2):
        reply = rpc({"jsonrpc": "2.0", "id": request_id + attempt, "method": method,
                     "params": {**params, "_meta": envelope(version)}})
        error = reply.get("error")
        if not error or error["code"] != -32022:
            return version, reply
        mutual = [v for v in CLIENT_SUPPORTS if v in error["data"]["supported"]]
        if not mutual:
            raise RuntimeError(f"no common version: server supports {error['data']['supported']}")
        version = mutual[0]
        print(f"... server rejected {error['data']['requested']}, retrying with {version}\n")
    raise RuntimeError("negotiation did not converge")


version, reply = call_with_negotiation(1, "tools/call", {"name": "weather_current", "arguments": {"location": "Oslo"}})
print("negotiated:", version, "| result:", reply["result"]["content"][0]["text"])
proc.stdin.close()
exit_code = proc.wait(timeout=10)

Notice there was no handshake: the *same* `tools/call` was simply re-sent with a different `_meta` version and a new `id`.

### 4. The SDK's two connection modes, on the wire
`Client(..., mode="legacy")` forces the 2025 `initialize` handshake; the default `mode="auto"` probes with `server/discover` and falls back only if the reply is not modern evidence. Same dual-era server, two very different conversations.

In [ ]:
import os

from mcp import Client, StdioServerParameters


def tapped(server_key, log_name):
    log = workdir / log_name
    log.unlink(missing_ok=True)
    return log, StdioServerParameters(command=sys.executable, args=[str(workdir / "wiretap.py"), str(log), str(SERVERS[server_key])])


def print_wire(log, width=260):
    for line in log.read_text().splitlines():
        print(line[:width])


for mode in ["legacy", "auto"]:
    log, params = tapped("dual-era", f"wire_{mode}.log")
    async with Client(params, mode=mode) as client:
        result = await client.call_tool("weather_current", {"location": "Paris"})
        print(f"===== mode={mode!r}: negotiated {client.protocol_version}, "
              f"tools.listChanged={client.server_capabilities.tools.list_changed}")
    print_wire(log)
    print()

Read the two logs side by side: **legacy** sends `initialize`, then `notifications/initialized`, then bare requests with no `_meta`, and results lack `resultType`. **auto** sends `server/discover` first and every later request carries the full envelope; results carry `resultType`, `ttlMs`/`cacheScope` and `serverInfo`. (The SDK also issues a `tools/list` after `tools/call` to validate the tool's output schema — handy to know when reading logs.)

### 5. Auto mode against a legacy server: the probe and the fallback
Now point `mode="auto"` at the legacy server. The probe gets a non-modern error (`-32600 Server not initialized`), so the client falls back to `initialize` — without being told anything about the server in advance. The legacy result has no `resultType`; the client treats it as `"complete"`, as the spec requires.

In [ ]:
log, params = tapped("legacy", "wire_auto_vs_legacy.log")
async with Client(params) as client:   # mode="auto" is the default
    result = await client.call_tool("weather_current", {"location": "Paris"})
    print("negotiated:", client.protocol_version, "| server:", client.server_info.name)
    print("result:", result.content[0].text, "| resultType seen by the app:", result.result_type)
print()
print_wire(log)

### 6. The compatibility matrix, live
Every combination of client mode and server era. `mode="2026-07-28"` pins a modern version without probing — the SDK's "modern-only client".

In [ ]:
async def attempt(mode, server_key):
    params = StdioServerParameters(command=sys.executable, args=[str(SERVERS[server_key])])
    try:
        async with Client(params, mode=mode, read_timeout_seconds=5) as client:
            result = await client.call_tool("weather_current", {"location": "Paris"})
            return f"works ({client.protocol_version})"
    except Exception as exc:
        while isinstance(exc, BaseExceptionGroup):   # unwrap anyio task-group wrappers
            exc = exc.exceptions[0]
        return f"FAILS: {str(exc)[:55]}"


modes = {"legacy": "legacy client", "auto": "dual-era client", "2026-07-28": "modern-only client"}
for mode, label in modes.items():
    for server_key in SERVERS:
        outcome = await attempt(mode, server_key)
        print(f"{label:<19} x {server_key + ' server':<19} -> {outcome}")
    print()

Compare with the spec's matrix: legacy × modern-only fails (and our server's error names the supported version — the only diagnostic a legacy client can show); modern-only × legacy fails; the dual-era client works everywhere.

### 7. Deprecation and removal in the SDK
Tier 1 SDKs must mark deprecated or removed API surface and should warn at runtime. `ping` was **removed** in 2026-07-28: the SDK still exposes `send_ping` for legacy connections, flags it with a warning, and a modern server answers `-32601 Method not found`.

In [ ]:
import warnings

for mode in ["legacy", "auto"]:
    params = StdioServerParameters(command=sys.executable, args=[str(SERVERS["dual-era"])])
    async with Client(params, mode=mode) as client:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            try:
                await client.send_ping()
                outcome = "pong (empty result)"
            except Exception as exc:
                outcome = f"error: {exc}"
        print(f"mode={mode:<7} protocol={client.protocol_version}  ping -> {outcome}")
        for w in caught:
            print(f"   {w.category.__name__}: {w.message}")

### 8. A migration linter for 2025-era traffic
Engineers inherit old code and logs. Encode the changelog and the deprecated registry as data, then scan the legacy wire log captured in section 4 (plus a few extra snippets) for anything removed or deprecated in 2026-07-28.

In [ ]:
RULES = [  # (needle, state, what to do instead)
    ('"method":"initialize"', "removed", "per-request _meta envelope; optional server/discover"),
    ("notifications/initialized", "removed", "nothing: there is no handshake"),
    ('"method":"ping"', "removed", "no protocol replacement (removed, not deprecated)"),
    ("logging/setLevel", "removed", "io.modelcontextprotocol/logLevel in each request's _meta"),
    ("resources/subscribe", "removed", "subscriptions/listen with resourceSubscriptions"),
    ("Mcp-Session-Id", "removed", "no sessions: server-minted handles passed as tool arguments"),
    ("Last-Event-ID", "removed", "re-issue the request with a new request ID"),
    ("notifications/roots/list_changed", "removed", "pass paths via tool parameters / config"),
    ("tasks/result", "removed", "Tasks extension: poll tasks/get"),
    ("notifications/elicitation/complete", "removed", "client retries the original request (MRTR)"),
    ("sampling/createMessage", "deprecated", "call an LLM provider API directly (SEP-2577)"),
    ("roots/list", "deprecated", "pass directories via tool parameters, resource URIs or config"),
    ("notifications/message", "deprecated", "log to stderr or use OpenTelemetry"),
    ('"includeContext":"thisServer"', "deprecated", 'omit includeContext or use "none"'),
    ("registration_endpoint", "deprecated", "Client ID Metadata Documents instead of DCR"),
]


def lint(lines):
    findings = {}
    for line in lines:
        compact = line.replace(" ", "")
        for needle, state, fix in RULES:
            if needle.replace(" ", "") in compact:
                findings.setdefault(needle, (state, fix, 0))
                s, f, n = findings[needle]
                findings[needle] = (s, f, n + 1)
    return findings


legacy_wire = (workdir / "wire_legacy.log").read_text().splitlines()
extra = ['POST /mcp  Mcp-Session-Id: 1868a90c', '{"method": "logging/setLevel", "params": {"level": "info"}}',
         '{"method": "sampling/createMessage", "params": {"includeContext": "thisServer"}}']
for needle, (state, fix, count) in lint(legacy_wire + extra).items():
    print(f"[{state.upper():<10}] {needle:<34} x{count}  -> {fix}")

### 9. When can a deprecated feature disappear?
The window is measured from the **release** of the deprecating revision; the earliest removal is the first revision released as Current on or after the window ends. The release dates below after 2026-07-28 are **invented** for the exercise.

In [ ]:
from datetime import date


def add_months(d, months):
    y, m = divmod(d.month - 1 + months, 12)
    return date(d.year + y, m + 1, min(d.day, 28))


def earliest_removal(deprecated_in, releases, months=12, days=None):
    """First release on/after the window ends. `days` models an expedited (security) window, min 90."""
    if days is not None:
        assert days >= 90, "expedited removal still needs at least ninety days"
        window_end = date.fromordinal(deprecated_in.toordinal() + days)
    else:
        assert months >= 12, "the standard window is at least twelve months"
        window_end = add_months(deprecated_in, months)
    return window_end, next((r for r in sorted(releases) if r >= window_end), None)


hypothetical_releases = [date(2026, 12, 15), date(2027, 6, 30), date(2027, 11, 18), date(2028, 5, 2)]
deprecated = date(2026, 7, 28)   # Roots, Sampling, Logging, DCR
for label, kwargs in [("standard 12 months", {}), ("SEP asks for 18 months", {"months": 18}),
                      ("expedited, 120 days", {"days": 120})]:
    end, release = earliest_removal(deprecated, hypothetical_releases, **kwargs)
    print(f"{label:<24} window ends {end} -> earliest removal in revision released {release}")

Note how the 12-month case lands on 2027-11-18, not 2027-07-28: removal only happens *in a release*, and the 2027-06-30 one came too early.

### Your turn
1. In section 3, change the dual-era server for the **modern-only** server and make `CLIENT_SUPPORTS = ["2027-03-01"]` only. Make `call_with_negotiation` surface a helpful error that quotes both version lists.
2. In section 5, edit `legacy_server.py` so it **never answers** requests before `initialize` (delete the `-32600` reply). Run `Client(params, read_timeout_seconds=3)` and observe how long `mode="auto"` takes to fall back. Which spec sentence covers this case?
3. Extend the linter in section 8 with three rules for changes that are *not* method names: the old resource-not-found code `-32002`, the old `UnsupportedProtocolVersion` code `-32004`, and a `tools/call` result that lacks `resultType`. What should each be migrated to?
4. Write a one-paragraph answer to "Our clients are all on 2025-11-25 SDKs. Can we deploy a modern-only server?" using the compatibility matrix.